# MC_Maze — load & explore

**Capstone, step 1: get the data into shape and understand it.**

We load the MC_Maze dataset (macaque M1 / PMd during delayed reaches; Churchland &
Shenoy labs) via [`nlb_tools`](https://github.com/neurallatents/nlb_tools) and build
a small set of **base loader functions** that turn the raw NWB file into:

- **binned spikes** (137 held-in neurons),
- **reach conditions** (per-trial reach target / direction),
- **hand velocity** (and other kinematics), trial-aligned to movement onset.

> Data source: DANDI dandiset **000128**, file
> `sub-Jenkins_ses-full_desc-train_behavior+ecephys.nwb` (~690 MB, gitignored).

### Prediction (to write together, *before* any tuning plot)
> _TODO — before plotting a single tuning curve, write one prediction here: what
> should a motor-cortex neuron's firing-rate-vs-reach-direction curve look like, and
> why?_

### Roadmap
1. **(this notebook)** load → binned spikes + reach conditions + hand velocity.
2. Understand the data (guided): sanity-check kinematics, the reach-direction
   definition, per-neuron activity.
3. Per-neuron directional **tuning curves**.
4. PCA of the population → reach-direction **decoder**.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nlb_tools.nwb_interface import NWBDataset

# --- config --------------------------------------------------------------
BIN_WIDTH_MS = 20      # spike-count bin size (native NWB resolution is 1 ms)


def _find_train_nwb():
    """Locate the MC_Maze train file whether run from ./notebooks or repo root."""
    rel = Path("data") / "000128" / "sub-Jenkins" / \
        "sub-Jenkins_ses-full_desc-train_behavior+ecephys.nwb"
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / rel).exists():
            return base / rel
    raise FileNotFoundError(
        f"Could not find {rel} — run the download step in the README.")


TRAIN_NWB = _find_train_nwb()
print("train file:", TRAIN_NWB)
print(f"size: {TRAIN_NWB.stat().st_size / 1e6:.0f} MB")

## 1. Load the raw dataset (1 ms bins)

`NWBDataset` reads the NWB file into two objects:

- **`.data`** — a continuous, time-indexed DataFrame (1 ms) with a column
  MultiIndex `(signal, channel)`: `spikes`, `heldout_spikes`, `hand_pos`,
  `hand_vel`, `cursor_pos`, `eye_pos`.
- **`.trial_info`** — one row per trial (targets, alignment events, success).

We only use the 137 held-in `spikes`; `heldout_spikes` exist for the NLB
co-smoothing benchmark and are ignored here.

In [6]:
def load_mc_maze(nwb_path):
    """Load the MC_Maze train file into an nlb_tools ``NWBDataset``.

    ``.data`` is continuous at 1 ms (spikes + hand/cursor/eye kinematics);
    ``.trial_info`` has one row per trial (targets, alignment events, success).
    """
    return NWBDataset(str(nwb_path))


dataset = load_mc_maze(TRAIN_NWB)

print("native bin width (ms):", dataset.bin_width)
print("continuous samples   :", dataset.data.shape[0], "(1 ms each)")
print("held-in neurons      :", dataset.data["spikes"].shape[1])
print("trials               :", len(dataset.trial_info))
print("signals              :", list(dataset.data.columns.get_level_values(0).unique()))

/Users/prateekkarkare/Desktop/Personal/Projects/motor-cortex-population-dynamics/.venv/lib/python3.9/site-packages/hdmf/spec/namespace.py:620: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.4.0, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


native bin width (ms): 1
continuous samples   : 6952301 (1 ms each)
held-in neurons      : 137
trials               : 2295
signals              : ['cursor_pos', 'eye_pos', 'hand_pos', 'hand_vel', 'heldout_spikes', 'spikes']


## 2. Bin the spikes

Resample from 1 ms to `BIN_WIDTH_MS`. Spikes are **summed** per bin (counts);
continuous signals (hand velocity, etc.) are **averaged**. This mutates the
dataset in place — re-run the load cell to get back to 1 ms.

In [ ]:
def bin_spikes(ds, bin_width_ms=BIN_WIDTH_MS):
    """Resample ``ds`` to ``bin_width_ms`` bins (in place) and return it.

    nlb_tools 0.0.4 targets pandas <= 1.3.4. Two things break under pandas >= 1.5,
    both fixed here:
      1. ``resample`` ends with ``data.index.freq = "<n>ms"``, which newer pandas
         validates strictly and rejects (the data is already rebinned by then).
      2. Without a proper index ``freq``, ``make_trial_data``'s ``slice_indexer``
         (slicing by non-bin-edge event times) fails.
    So we catch the freq error and rebuild a clean, regular ``TimedeltaIndex``.
    """
    if ds.bin_width == bin_width_ms:
        return ds
    n_before = ds.data.shape[0]
    try:
        ds.resample(bin_width_ms)
    except ValueError:
        assert ds.data.shape[0] < n_before, "resample did not actually rebin"
        ds.bin_width = bin_width_ms
    ds.data.index = pd.timedelta_range(
        start=ds.data.index[0], periods=len(ds.data),
        freq=f"{bin_width_ms}ms", name="clock_time",
    )
    assert ds.bin_width == bin_width_ms
    return ds


bin_spikes(dataset, BIN_WIDTH_MS)

# sanity: median spacing of the binned index should equal the bin width
_step_ms = np.median(np.diff(dataset.data.index.total_seconds())) * 1000
print("bin width (ms):", dataset.bin_width, f"(median index step {_step_ms:.1f} ms)")
print("binned samples:", dataset.data.shape[0])
print("spikes block  :", dataset.data["spikes"].shape)

bin width (ms): 20 (median index step 20.0 ms)
binned samples: 347616
spikes block  : (347616, 137)


## 3. Reach conditions per trial

Each trial shows up to 3 targets (`target_pos`); `active_target` indexes the one
actually reached to. The reach **direction** is the angle of that target from the
center hold.

> In the maze task the *path* can curve around barriers, so "target direction" and
> "instantaneous hand direction" are not identical — which one counts as *reach
> direction* is a modelling choice we'll revisit before building tuning curves.
> This function surfaces the target-based angle plus the raw fields.

In [8]:
def reach_conditions(ds):
    """One row per trial describing *where the reach went*.

    The reached target is ``target_pos[active_target]``; ``reach_angle_deg`` is
    the direction of that target from center (target-based reach direction).
    """
    ti = ds.trial_info
    target_pos = ti["target_pos"].to_numpy()
    active = ti["active_target"].to_numpy()
    xy = np.array([np.asarray(target_pos[i], dtype=float)[active[i]]
                   for i in range(len(ti))])
    x, y = xy[:, 0], xy[:, 1]
    return pd.DataFrame({
        "trial_id": ti["trial_id"].to_numpy(),
        "target_x": x,
        "target_y": y,
        "reach_angle_deg": np.degrees(np.arctan2(y, x)),
        "reach_dist": np.hypot(x, y),
        "trial_type": ti["trial_type"].to_numpy(),
        "maze_id": ti["maze_id"].to_numpy(),
        "num_targets": ti["num_targets"].to_numpy(),
        "success": ti["success"].to_numpy(),
        "move_onset_time": ti["move_onset_time"].to_numpy(),
    })


conditions = reach_conditions(dataset)
print(conditions.shape)
conditions.head()

(2295, 10)


,trial_id,target_x,target_y,reach_angle_deg,reach_dist,trial_type,maze_id,num_targets,success,move_onset_time
0,0,118.0,72.0,31.390268,138.231690,25,84,3,True,0 days 00:00:01.905000
1,1,-116.0,-5.0,-177.531882,116.107709,3,3,1,True,0 days 00:00:05.280000
2,2,-82.0,-86.0,-133.636072,118.827606,22,66,1,True,0 days 00:00:08.346000
3,3,2.0,82.0,88.602819,82.024387,29,100,3,True,0 days 00:00:11.752000
4,4,27.0,82.0,71.774925,86.330759,21,65,1,True,0 days 00:00:14.507000


## 4. Trial-aligned spikes + hand velocity

`make_trial_data` cuts the continuous data into per-trial segments aligned to a
task event (here **movement onset**) within a time window. Bin edges are relative
to the alignment event, so trials become directly comparable — the form we'll use
to compute per-neuron firing rates by reach direction.

In [9]:
def make_trial_spikes(ds, align_field="move_onset_time", window_ms=(-200, 600)):
    """Trial-aligned binned data around a task event.

    Returns the nlb_tools trial DataFrame: every trial's binned spikes and
    kinematics (incl. ``hand_vel``), clipped to ``window_ms`` around
    ``align_field`` (e.g. movement onset), plus meta columns like ``trial_id``
    and ``align_time``.
    """
    return ds.make_trial_data(align_field=align_field, align_range=window_ms)


trials = make_trial_spikes(dataset)
bins_per_trial = int((600 - (-200)) / dataset.bin_width)
print("trial-aligned rows :", trials.shape[0])
print("signals (level 0)  :", list(trials.columns.get_level_values(0).unique()))
print("expected bins/trial:", bins_per_trial, f"(-200..600 ms @ {dataset.bin_width} ms)")
trials.head()

KeyError: Timedelta('0 days 00:00:01.705000')

## Next (guided)

Base loaders are in place: `load_mc_maze` → `bin_spikes` → `reach_conditions` +
`make_trial_spikes`. Before any tuning curve we'll, together:

1. **write the prediction** (top cell),
2. **understand the data** — check hand velocity vs reach angle, look at a few
   neurons' activity, decide how to define reach direction and the rate window,
3. then build the **directional tuning curves** for 4–5 neurons.

_No tuning-curve code yet — that's the next, guided step._